In [1]:
!pip install tokenizers


In [2]:
# Read your train.txt and write a clean version for tokenizer training
with open("train.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Save as plain corpus for tokenizer trainer
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"✅ Corpus ready — {len(text):,} characters")

✅ Corpus ready — 112,857,375 characters


In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

# Special tokens
special_tokens = [
    "<pad>",   # 0
    "<bos>",   # 1
    "<eos>",   # 2
    "<unk>",   # 3
    "<question>",   # 4
    "</question>",  # 5
    "<answer>",     # 6
    "</answer>",    # 7
]

# Build tokenizer
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=16000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("Training BPE tokenizer on medical corpus...")
tokenizer.train(files=["corpus.txt"], trainer=trainer)

# Add BOS/EOS wrapping automatically
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", tokenizer.token_to_id("<bos>")),
        ("<eos>", tokenizer.token_to_id("<eos>")),
    ],
)

# Save tokenizer
tokenizer.save("medical_tokenizer.json")
print(f"✅ Tokenizer trained! Vocab size: {tokenizer.get_vocab_size()}")

# Quick test
test = "<question> What causes tachycardia and myocardial infarction? </question>"
encoded = tokenizer.encode(test)
print(f"\nTest encoding:")
print(f"  Input : {test}")
print(f"  Tokens: {encoded.tokens}")
print(f"  IDs   : {encoded.ids}")

Training BPE tokenizer on medical corpus...
✅ Tokenizer trained! Vocab size: 16000

Test encoding:
  Input : <question> What causes tachycardia and myocardial infarction? </question>
  Tokens: ['<bos>', '<question>', 'What', 'causes', 'tachycardia', 'and', 'myocardial', 'infarction', '?', '</question>', '<eos>']
  IDs   : [1, 4, 1379, 903, 3765, 280, 3747, 3377, 44, 5, 2]


In [4]:
import numpy as np

def encode_file(input_path, output_path, tokenizer):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Split into individual Q&A pairs
    pairs = text.strip().split("\n\n")
    print(f"  Encoding {len(pairs):,} pairs from {input_path}...")

    all_ids = []
    for pair in pairs:
        pair = pair.strip()
        if pair:
            encoded = tokenizer.encode(pair)
            all_ids.extend(encoded.ids)

    arr = np.array(all_ids, dtype=np.uint16)
    arr.tofile(output_path)
    print(f"  ✅ Saved {len(arr):,} tokens → {output_path}")
    return len(arr)

print("Encoding train.txt...")
train_tokens = encode_file("train.txt", "train.bin", tokenizer)

print("\nEncoding val.txt...")
val_tokens = encode_file("val.txt", "val.bin", tokenizer)

print(f"\n✅ Done!")
print(f"  train.bin : {train_tokens:,} tokens")
print(f"  val.bin   : {val_tokens:,} tokens")
print(f"  Total     : {train_tokens + val_tokens:,} tokens")

Encoding train.txt...
  Encoding 189,075 pairs from train.txt...
  ✅ Saved 23,833,856 tokens → train.bin

Encoding val.txt...
  Encoding 21,017 pairs from val.txt...
  ✅ Saved 2,671,488 tokens → val.bin

✅ Done!
  train.bin : 23,833,856 tokens
  val.bin   : 2,671,488 tokens
  Total     : 26,505,344 tokens


In [5]:
from tokenizers import Tokenizer
import numpy as np

# Reload tokenizer and test
tok = Tokenizer.from_file("medical_tokenizer.json")

print("=== Tokenizer Stats ===")
print(f"Vocab size : {tok.get_vocab_size():,}")
print(f"Special tokens:")
for st in ["<pad>", "<bos>", "<eos>", "<unk>", "<question>", "</question>", "<answer>", "</answer>"]:
    print(f"  {st:15s} → ID {tok.token_to_id(st)}")

print("\n=== Binary Files ===")
train_bin = np.fromfile("train.bin", dtype=np.uint16)
val_bin   = np.fromfile("val.bin",   dtype=np.uint16)
print(f"train.bin : {len(train_bin):,} tokens")
print(f"val.bin   : {len(val_bin):,} tokens")

print("\n=== Medical Word Test ===")
medical_words = ["tachycardia", "myocardial", "infarction", "subluxation", "ophthalmology", "hypertension"]
for word in medical_words:
    tokens = tok.encode(word).tokens
    print(f"  {word:20s} → {tokens}")

print("\n✅ Tokenizer ready for model training!")

=== Tokenizer Stats ===
Vocab size : 16,000
Special tokens:
  <pad>           → ID 0
  <bos>           → ID 1
  <eos>           → ID 2
  <unk>           → ID 3
  <question>      → ID 4
  </question>     → ID 5
  <answer>        → ID 6
  </answer>       → ID 7

=== Binary Files ===
train.bin : 23,833,856 tokens
val.bin   : 2,671,488 tokens

=== Medical Word Test ===
  tachycardia          → ['<bos>', 'tachycardia', '<eos>']
  myocardial           → ['<bos>', 'myocardial', '<eos>']
  infarction           → ['<bos>', 'infarction', '<eos>']
  subluxation          → ['<bos>', 'subluxation', '<eos>']
  ophthalmology        → ['<bos>', 'ophthalmology', '<eos>']
  hypertension         → ['<bos>', 'hypertension', '<eos>']

✅ Tokenizer ready for model training!
